# See which 'same statement' the kernel accepts

*Adversarial Mathematics From First Principles*, [chapter 22](https://programmer.ie/books/math/22-chapter/) · [worked example](https://programmer.ie/examples/math/22-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

There are at least five things "these two statements are the same" could
mean, from equal text to meaning the same thing. The laboratory's binding
check asks Lean to accept a declaration as a proof of the pinned statement:
that checks definitional equality, and nothing weaker. This core-Lean file
shows where that line falls.

## Run it

Colab has no Lean, so the first cell downloads the official Lean 4.35.0 release (core only, about 600 MB, a minute or two). The second writes the file and the third checks it. The two type mismatches are the expected result.

In [1]:
!apt-get -qq install -y zstd > /dev/null
!test -x lean-4.35.0-linux/bin/lean || curl -sSfL https://github.com/leanprover/lean4/releases/download/v4.35.0/lean-4.35.0-linux.tar.zst | tar --zstd -x
!lean-4.35.0-linux/bin/lean --version

Lean (version 4.35.0, x86_64-unknown-linux-gnu, commit eb6cafb7290e8bd5f2937522d8d804f58f02bafc, Release)


In [2]:
%%writefile statement_identity.lean
-- Statement identity: what the binding check accepts, and what it does not (core Lean 4).

theorem proved : ∀ n : Nat, 0 + n = n := fun n => Nat.zero_add n

-- 1-3. Bound variables renamed, or notation unfolded: definitionally the same. Accepted.
example : ∀ m : Nat, 0 + m = m := proved
example : ∀ k : Nat, Nat.add 0 k = k := proved

-- 4. Logically equivalent, but only with a proof: the binding check refuses it.
theorem exists_nonzero : ∃ n : Nat, n ≠ 0 := ⟨1, by decide⟩
example : ¬ ∀ n : Nat, n = 0 := exists_nonzero

-- The equivalence is a theorem in its own right, not something the binding sees.
example : ¬ ∀ n : Nat, n = 0 := fun h => absurd (h 1) (by decide)

-- 5. A weaker statement is a different statement. Refused.
theorem small_only : ∀ n : Nat, n ≤ 10 → 0 + n = n := fun n _ => Nat.zero_add n
example : ∀ n : Nat, 0 + n = n := small_only

Writing statement_identity.lean


In [3]:
!lean-4.35.0-linux/bin/lean statement_identity.lean

statement_identity.lean:11:32: error: Type mismatch
  exists_nonzero
has type
  ∃ n, n ≠ 0
but is expected to have type
  ¬∀ (n : Nat), n = 0
statement_identity.lean:18:34: error: Type mismatch
  small_only
has type
  ∀ (n : Nat), n ≤ 10 → 0 + n = n
but is expected to have type
  ∀ (n : Nat), 0 + n = n


## What the output shows

Renaming the bound variable, or writing `Nat.add 0 k` for `0 + k`, is
definitionally the same statement, and the binding is accepted silently. A
proof of `∃ n, n ≠ 0` is not accepted as a proof of `¬ ∀ n, n = 0`, although
the two are equivalent: that needs a proof, given on the next line. A weaker
statement is refused. The two errors are the expected output.

## Not shown

- Logical equivalence is deliberately not attempted by the binding check; it would need a proof for each pair.
- Whether a statement means what the claim says (the fifth sense) is the correspondence, which only a person can approve.

## Challenge it

Examine each binding in the file and classify it by the five senses of "the
same statement". Predict which Lean accepts, then run it. Find two statements
that are definitionally equal but would look different to a reader, and two
that look the same but are not definitionally equal.